# LeetCode 3915 — Maximum Sum of Alternating Subsequence With Distance at Least K bằng Python

- Link: https://leetcode.com/problems/maximum-sum-of-alternating-subsequence-with-distance-at-least-k/
- Độ khó: **Hard**
- Chủ đề: Quy hoạch động trên dãy con, nén tọa độ, Fenwick tree (BIT) lấy max, segment tree, kỹ thuật "chèn trễ k bước"

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def maxAlternatingSum(self, nums: list[int], k: int) -> int:
```

## 1. Mục tiêu học

1. Định nghĩa DP hai trạng thái cho dãy con **luân phiên** (kết thúc ở "đáy" hay "đỉnh").
2. Biến điều kiện "phần tử trước **nhỏ hơn / lớn hơn**" thành **truy vấn max tiền tố / hậu tố** trên trục giá trị (sau khi nén tọa độ).
3. Xử lý điều kiện **khoảng cách chỉ số ≥ k** bằng kỹ thuật **chèn trễ**: chỉ đưa vị trí `i - k` vào cấu trúc dữ liệu khi xử lý `i`.
4. Cài đặt Fenwick tree max (và cách lật trục để có max hậu tố), cùng phương án segment tree.
5. Kiểm thử đối chiếu với vét cạn và DP `O(n²)`.

## 2. Đề bài & ví dụ

**Phát biểu lại.** Cho mảng `nums` độ dài `n` và số `k`. Chọn một dãy con với chỉ số `i_1 < i_2 < … < i_m` sao cho:

- hai chỉ số liên tiếp cách nhau **ít nhất `k`**: `i_{t+1} - i_t ≥ k`;
- các giá trị được chọn **luân phiên ngặt**: `a < b > c < …` hoặc `a > b < c > …` (dãy 1 phần tử luôn hợp lệ).

Điểm của dãy = tổng các giá trị được chọn. Tìm điểm lớn nhất.

**Ví dụ 1:** `nums = [5,4,2], k = 2` → `7` (chọn chỉ số 0 và 2: `5 > 2`).

**Ví dụ 2:** `nums = [3,5,4,2,4], k = 1` → `14` (chọn `3 < 5 > 2 < 4`).

**Ví dụ 3:** `nums = [5], k = 1` → `5`.

**Ràng buộc:** `1 ≤ n ≤ 10^5`, `1 ≤ nums[i] ≤ 10^5`, `1 ≤ k ≤ n`.

In [1]:
from bisect import bisect_left
import random, time

examples = [([5, 4, 2], 2, 7), ([3, 5, 4, 2, 4], 1, 14), ([5], 1, 5)]
for nums, k, exp in examples:
    print(nums, "k =", k, "-> kỳ vọng", exp)

[5, 4, 2] k = 2 -> kỳ vọng 7
[3, 5, 4, 2, 4] k = 1 -> kỳ vọng 14
[5] k = 1 -> kỳ vọng 5


## 3. Kiến thức nền

### 3.1. Kiểm tra dãy luân phiên ngặt

Dãy `b` luân phiên nếu mọi cặp kề nhau khác nhau và chiều so sánh đổi dấu liên tục. Cách viết gọn: với mọi `t`, `(b[t] - b[t-1]) * (b[t+1] - b[t]) < 0`.

In [2]:
def is_alternating(b):
    if any(b[t] == b[t + 1] for t in range(len(b) - 1)):
        return False
    return all((b[t] - b[t - 1]) * (b[t + 1] - b[t]) < 0 for t in range(1, len(b) - 1))

print(is_alternating([3, 5, 2, 4]), is_alternating([1, 2, 3]), is_alternating([7]), is_alternating([4, 4]))

True False True False


### 3.2. Nén tọa độ (coordinate compression)

Giá trị tới `10^5` thì dùng trực tiếp được, nhưng thói quen tốt là nén: sắp các giá trị phân biệt, gán **hạng** `1..m`. "Giá trị nhỏ hơn `x`" ⇔ "hạng `< rank[x]`".

In [3]:
nums = [30, 10, 30, 20]
vals = sorted(set(nums))
rank = {v: i + 1 for i, v in enumerate(vals)}
print(vals, rank, [rank[x] for x in nums])

[10, 20, 30] {10: 1, 20: 2, 30: 3} [3, 1, 3, 2]


### 3.3. Fenwick tree lấy max (tiền tố và hậu tố)

- `update(i, val)`: `t[i] = max(t[i], val)` lan lên các nút cha. **Chỉ dùng khi giá trị tại mỗi vị trí chỉ tăng.**
- `query(i)`: max trên `[1..i]`.
- Muốn **max hậu tố** `[i..m]`, lật trục: vị trí `i` lưu ở chỉ số `m + 1 - i`, khi đó hậu tố trở thành tiền tố.

In [4]:
class BITMax:
    def __init__(self, n):
        self.n = n
        self.c = [0] * (n + 1)
    def update(self, x, val):
        while x <= self.n:
            if self.c[x] < val:
                self.c[x] = val
            x += x & -x
    def query(self, x):
        res = 0
        while x > 0:
            if self.c[x] > res:
                res = self.c[x]
            x -= x & -x
        return res

b = BITMax(5)
b.update(2, 7); b.update(4, 3)
print("max [1..1] =", b.query(1), "| max [1..3] =", b.query(3), "| max [1..5] =", b.query(5))
# hậu tố: lưu vị trí i ở m+1-i
m = 5; rb = BITMax(m)
rb.update(m + 1 - 2, 7); rb.update(m + 1 - 4, 3)
print("max hậu tố [3..5] =", rb.query(m + 1 - 3), "| max hậu tố [2..5] =", rb.query(m + 1 - 2))

max [1..1] = 0 | max [1..3] = 7 | max [1..5] = 7
max hậu tố [3..5] = 3 | max hậu tố [2..5] = 7


### 3.4. Kỹ thuật "chèn trễ" (sliding insertion)

Khi điều kiện là `j ≤ i - k`, ta duyệt `i` tăng dần và **ngay trước khi xử lý `i`** mới chèn vị trí `j = i - k` vào cấu trúc dữ liệu. Khi đó cấu trúc chứa đúng các `j ∈ [0, i - k]`.

In [5]:
k = 2
inserted = []
for i in range(6):
    if i >= k:
        inserted.append(i - k)
    print(f"xử lý i={i}: cấu trúc chứa các j = {inserted}")

xử lý i=0: cấu trúc chứa các j = []
xử lý i=1: cấu trúc chứa các j = []
xử lý i=2: cấu trúc chứa các j = [0]
xử lý i=3: cấu trúc chứa các j = [0, 1]
xử lý i=4: cấu trúc chứa các j = [0, 1, 2]
xử lý i=5: cấu trúc chứa các j = [0, 1, 2, 3]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Gọi phần tử cuối của dãy là **đáy** nếu phần tử tiếp theo (nếu có) phải **lớn hơn** nó, và **đỉnh** nếu phần tử tiếp theo phải **nhỏ hơn**. Mỗi phần tử trong dãy luân phiên là đáy hoặc đỉnh, xen kẽ nhau.

Tính tay ví dụ 2: `nums = [3,5,4,2,4]`, `k = 1`. Dãy tối ưu `3 < 5 > 2 < 4`:

- `3` là đáy (tiếp theo lớn hơn), `5` là đỉnh, `2` là đáy, `4` là đỉnh (hoặc phần tử cuối).
- Tổng `14`.

Một phần tử **cuối dãy** có thể coi là đáy hoặc đỉnh tùy ý (không còn phần tử sau). Dãy 1 phần tử vừa là đáy vừa là đỉnh.

Vậy với mỗi vị trí `i`, ta muốn biết:

- `down[i]`: tổng lớn nhất của dãy hợp lệ **kết thúc tại `i` và `i` là đáy** — phần tử trước (nếu có) là **đỉnh lớn hơn** `nums[i]`.
- `up[i]`: tổng lớn nhất **kết thúc tại `i` và `i` là đỉnh** — phần tử trước (nếu có) là **đáy nhỏ hơn**.

## 5. Cách 1 — Brute Force: thử mọi dãy con

Duyệt mọi tập chỉ số (bitmask), kiểm tra khoảng cách và tính luân phiên, lấy tổng lớn nhất. `O(2^n · n)`.

In [6]:
def brute(nums, k):
    n = len(nums)
    best = 0
    for mask in range(1, 1 << n):
        idx = [i for i in range(n) if mask >> i & 1]
        if any(idx[t + 1] - idx[t] < k for t in range(len(idx) - 1)):
            continue
        vals = [nums[i] for i in idx]
        if is_alternating(vals):
            best = max(best, sum(vals))
    return best

for nums, k, exp in examples:
    print(brute(nums, k), "kỳ vọng", exp)

7 kỳ vọng 7
14 kỳ vọng 14
5 kỳ vọng 5


**Nút thắt:** số dãy con là `2^n`. Nhưng thông tin cần để nối thêm phần tử chỉ là **(phần tử cuối, vai trò đáy/đỉnh)** ⇒ quy hoạch động.

## 6. Ý tưởng thuật toán / bước đột phá

**Công thức DP:**

- `down[i] = nums[i] + max(0, max{ up[j] : j ≤ i - k, nums[j] > nums[i] })`
- `up[i]   = nums[i] + max(0, max{ down[j] : j ≤ i - k, nums[j] < nums[i] })`

(`max(0, …)` ứng với việc `i` là phần tử đầu tiên.) Đáp án = `max` mọi `down[i]`, `up[i]`.

**Đột phá:** hai cái `max{…}` là **truy vấn max theo khoảng giá trị**:

- `nums[j] > nums[i]` ⇒ **max hậu tố** trên trục hạng giá trị (của mảng `up`).
- `nums[j] < nums[i]` ⇒ **max tiền tố** (của mảng `down`).

Điều kiện `j ≤ i - k` ⇒ **chèn trễ**: trước khi xử lý `i`, chèn vị trí `i - k`. Mỗi bước `O(log m)` ⇒ tổng `O(n log n)`.

Quy tắc nhớ: *"đáy hỏi đỉnh lớn hơn, đỉnh hỏi đáy nhỏ hơn"*.

## 7. Các cách tiếp theo

- **Cách 2:** DP `O(n²)` trực tiếp theo công thức.
- **Cách 3:** DP + **hai Fenwick tree max** (tiền tố cho `down`, hậu tố — lật trục — cho `up`) + chèn trễ. Lời giải nộp chính.
- **Cách 4:** DP + **segment tree** max trên trục giá trị (không cần lật trục, truy vấn đoạn bất kỳ).

### 7.1. Cách 2 — DP `O(n²)`

In [7]:
def dp_quadratic(nums, k):
    n = len(nums)
    down = [0] * n
    up = [0] * n
    for i in range(n):
        bd = bu = 0
        for j in range(0, i - k + 1):
            if nums[j] > nums[i]:
                bd = max(bd, up[j])
            elif nums[j] < nums[i]:
                bu = max(bu, down[j])
        down[i] = nums[i] + bd
        up[i] = nums[i] + bu
    return max(max(down), max(up))

for nums, k, exp in examples:
    print(dp_quadratic(nums, k), "kỳ vọng", exp)

7 kỳ vọng 7
14 kỳ vọng 14
5 kỳ vọng 5


In [8]:
class SolutionQuadratic:
    def maxAlternatingSum(self, nums: list[int], k: int) -> int:
        return dp_quadratic(nums, k)

print([SolutionQuadratic().maxAlternatingSum(a, k) for a, k, _ in examples])

[7, 14, 5]


**Độ phức tạp:** `O(n²)` thời gian, `O(n)` bộ nhớ. `n = 10^5` ⇒ `5·10^9` bước — quá chậm.

### 7.2. Cách 3 — DP + hai Fenwick tree max (lời giải chính)

**Giải thích từng đoạn:**

1. Nén `nums` về hạng `1..m`.
2. `bit_down` (tiền tố): tại hạng `r` lưu `down[j]`. Truy vấn `query(r_i - 1)` = max `down[j]` với `nums[j] < nums[i]`.
3. `bit_up` (hậu tố, lật trục): tại chỉ số `m + 1 - r` lưu `up[j]`. Truy vấn `query(m - r_i)` = max `up[j]` với hạng `> r_i`.
4. Duyệt `i`: nếu `i ≥ k`, chèn `down[i-k]`, `up[i-k]`. Sau đó tính `down[i]`, `up[i]`.

Vì mọi giá trị đều dương và DP chỉ tăng tại mỗi vị trí của BIT (ta lấy `max`), Fenwick max dùng được.

In [9]:
def bit_solution(nums, k):
    vals = sorted(set(nums))
    m = len(vals)
    rank = {v: i + 1 for i, v in enumerate(vals)}
    bit_down, bit_up = BITMax(m), BITMax(m)
    n = len(nums)
    down, up = [0] * n, [0] * n
    ans = 0
    for i, x in enumerate(nums):
        if i >= k:
            j = i - k
            rj = rank[nums[j]]
            bit_down.update(rj, down[j])
            bit_up.update(m + 1 - rj, up[j])
        r = rank[x]
        down[i] = x + bit_up.query(m - r)       # j có hạng > r
        up[i] = x + bit_down.query(r - 1)       # j có hạng < r
        ans = max(ans, down[i], up[i])
    return ans

for nums, k, exp in examples:
    print(bit_solution(nums, k), "kỳ vọng", exp)

7 kỳ vọng 7
14 kỳ vọng 14
5 kỳ vọng 5


**Độ phức tạp Cách 3:** `O(n log n)` thời gian (sắp xếp + `2n` update + `2n` query), `O(n)` bộ nhớ.

### 7.3. Cách 4 — DP + segment tree max

Segment tree lặp (iterative, kích thước `2·size`) hỗ trợ `update(pos, val)` lấy max và `query(lo, hi)` max trên đoạn `[lo, hi)`. Không cần lật trục: truy vấn `(r, m)` cho hạng lớn hơn và `(0, r)` cho hạng nhỏ hơn (hạng 0-indexed).

In [10]:
class SegMax:
    def __init__(self, n):
        self.size = 1
        while self.size < n:
            self.size *= 2
        self.t = [0] * (2 * self.size)
    def update(self, pos, val):
        pos += self.size
        if self.t[pos] >= val:
            return
        self.t[pos] = val
        pos //= 2
        while pos:
            self.t[pos] = max(self.t[2 * pos], self.t[2 * pos + 1])
            pos //= 2
    def query(self, lo, hi):          # max trên [lo, hi)
        res = 0
        lo += self.size; hi += self.size
        while lo < hi:
            if lo & 1:
                res = max(res, self.t[lo]); lo += 1
            if hi & 1:
                hi -= 1; res = max(res, self.t[hi])
            lo //= 2; hi //= 2
        return res

def seg_solution(nums, k):
    vals = sorted(set(nums))
    m = len(vals)
    n = len(nums)
    rk = [bisect_left(vals, x) for x in nums]     # hạng 0-indexed
    seg_down, seg_up = SegMax(m), SegMax(m)
    down, up = [0] * n, [0] * n
    ans = 0
    for i in range(n):
        if i >= k:
            j = i - k
            seg_down.update(rk[j], down[j])
            seg_up.update(rk[j], up[j])
        r = rk[i]
        down[i] = nums[i] + seg_up.query(r + 1, m)
        up[i] = nums[i] + seg_down.query(0, r)
        ans = max(ans, down[i], up[i])
    return ans

for nums, k, exp in examples:
    print(seg_solution(nums, k), "kỳ vọng", exp)

7 kỳ vọng 7
14 kỳ vọng 14
5 kỳ vọng 5


In [11]:
class SolutionSegTree:
    def maxAlternatingSum(self, nums: list[int], k: int) -> int:
        return seg_solution(nums, k)

print([SolutionSegTree().maxAlternatingSum(a, k) for a, k, _ in examples])

[7, 14, 5]


**Độ phức tạp Cách 4:** `O(n log n)`, bộ nhớ `O(n)`. Hằng số lớn hơn Fenwick một chút nhưng linh hoạt hơn (truy vấn đoạn bất kỳ, ví dụ nếu đề thêm điều kiện "chênh lệch giá trị ≤ d").

## 8. TRACE từng bước

Trace Cách 3 trên ví dụ 2: `nums = [3,5,4,2,4]`, `k = 1`. Hạng: `2→1, 3→2, 4→3, 5→4`.

| i | x | chèn j | down[i] = x + max up(lớn hơn) | up[i] = x + max down(nhỏ hơn) |
|---|---|---|---|---|
| 0 | 3 | – | 3 | 3 |
| 1 | 5 | 0 | 5 + 0 = 5 | 5 + down[0]=3 → 8 |
| 2 | 4 | 1 | 4 + up[1]=8 → 12 | 4 + down[0]=3 → 7 |
| 3 | 2 | 2 | 2 + max(up)=8 → 10 | 2 + 0 = 2 |
| 4 | 4 | 3 | 4 + up[1]=8 → 12 | 4 + max(down[0]=3, down[3]=10) → 14 |

Đáp án `14` = `up[4]`: dãy `3 < 5 > 2 < 4`.

In [12]:
def trace_bit(nums, k):
    vals = sorted(set(nums)); m = len(vals)
    rank = {v: i + 1 for i, v in enumerate(vals)}
    bd, bu = BITMax(m), BITMax(m)
    n = len(nums); down, up = [0] * n, [0] * n
    for i, x in enumerate(nums):
        ins = "-"
        if i >= k:
            j = i - k; rj = rank[nums[j]]
            bd.update(rj, down[j]); bu.update(m + 1 - rj, up[j]); ins = str(j)
        r = rank[x]
        qa, qb = bu.query(m - r), bd.query(r - 1)
        down[i], up[i] = x + qa, x + qb
        print(f"i={i} x={x} hạng={r} chèn j={ins:<2} | max up(lớn hơn)={qa:<3} -> down={down[i]:<3} | max down(nhỏ hơn)={qb:<3} -> up={up[i]}")
    print("đáp án =", max(max(down), max(up)))

trace_bit([3, 5, 4, 2, 4], 1)

i=0 x=3 hạng=2 chèn j=-  | max up(lớn hơn)=0   -> down=3   | max down(nhỏ hơn)=0   -> up=3
i=1 x=5 hạng=4 chèn j=0  | max up(lớn hơn)=0   -> down=5   | max down(nhỏ hơn)=3   -> up=8
i=2 x=4 hạng=3 chèn j=1  | max up(lớn hơn)=8   -> down=12  | max down(nhỏ hơn)=3   -> up=7
i=3 x=2 hạng=1 chèn j=2  | max up(lớn hơn)=8   -> down=10  | max down(nhỏ hơn)=0   -> up=2
i=4 x=4 hạng=3 chèn j=3  | max up(lớn hơn)=8   -> down=12  | max down(nhỏ hơn)=10  -> up=14
đáp án = 14


In [13]:
# Trace với k = 2 (ví dụ 1): j phải cách ít nhất 2
trace_bit([5, 4, 2], 2)

i=0 x=5 hạng=3 chèn j=-  | max up(lớn hơn)=0   -> down=5   | max down(nhỏ hơn)=0   -> up=5
i=1 x=4 hạng=2 chèn j=-  | max up(lớn hơn)=0   -> down=4   | max down(nhỏ hơn)=0   -> up=4
i=2 x=2 hạng=1 chèn j=0  | max up(lớn hơn)=5   -> down=7   | max down(nhỏ hơn)=0   -> up=2
đáp án = 7


## 9. Code nộp LeetCode

In [14]:
class Solution:
    def maxAlternatingSum(self, nums: list[int], k: int) -> int:
        vals = sorted(set(nums))
        m = len(vals)
        rank = {v: i + 1 for i, v in enumerate(vals)}
        c_down = [0] * (m + 1)       # Fenwick max tiền tố trên down
        c_up = [0] * (m + 1)         # Fenwick max tiền tố trên up (trục lật)

        def upd(c, x, val):
            while x <= m:
                if c[x] < val:
                    c[x] = val
                x += x & -x

        def qry(c, x):
            res = 0
            while x > 0:
                if c[x] > res:
                    res = c[x]
                x -= x & -x
            return res

        n = len(nums)
        down = [0] * n
        up = [0] * n
        ans = 0
        for i, x in enumerate(nums):
            if i >= k:
                j = i - k
                rj = rank[nums[j]]
                upd(c_down, rj, down[j])
                upd(c_up, m + 1 - rj, up[j])
            r = rank[x]
            down[i] = x + qry(c_up, m - r)
            up[i] = x + qry(c_down, r - 1)
            if down[i] > ans:
                ans = down[i]
            if up[i] > ans:
                ans = up[i]
        return ans

sol = Solution()
for nums, k, exp in examples:
    got = sol.maxAlternatingSum(nums, k)
    print(nums, k, "->", got, "✔" if got == exp else "✘")

[5, 4, 2] 2 -> 7 ✔
[3, 5, 4, 2, 4] 1 -> 14 ✔
[5] 1 -> 5 ✔


## 10. Vì sao thuật toán đúng

1. **Phân rã:** mọi dãy hợp lệ có ≥ 2 phần tử, bỏ phần tử cuối vẫn hợp lệ. Nếu phần tử cuối `i` nhỏ hơn phần tử trước `j` thì `i` là đáy và `j` là đỉnh (phần tử sau `j` nhỏ hơn `j`), và ngược lại. Nên `down[i]`, `up[i]` được tính đúng bằng công thức truy hồi (quy nạp theo `i`).
2. **Không chồng lấn vai trò:** `j` là "đỉnh" nghĩa là dãy kết thúc ở `j` có bước cuối đi lên tới `j` (hoặc `j` đứng đầu). Giá trị `up[j]` đúng là tổng lớn nhất của các dãy như vậy, vì vậy nối thêm `i < nums[j]` giữ tính luân phiên.
3. **Bất biến của chèn trễ:** khi xử lý `i`, cấu trúc chứa đúng các `j ∈ [0, i-k]` (chèn `i-k` tại bước `i`, không chèn trùng). Vậy điều kiện khoảng cách được bảo đảm chính xác.
4. **Fenwick max:** mỗi ô chỉ được cập nhật bằng `max`, nên giá trị tiền tố luôn là max các giá trị đã chèn có hạng trong đoạn.

## 11. Phân tích Time / Space Complexity

Đặt `n = len(nums)`, `m` = số giá trị phân biệt (`m ≤ n`).

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn | `O(2^n · n)` | `O(n)` |
| 2. DP | `O(n²)` | `O(n)` |
| 3. DP + 2 Fenwick | `O(n log m)` | `O(n + m)` |
| 4. DP + segment tree | `O(n log m)` | `O(n + m)` |

## 12. Bộ test

In [15]:
def run_all(nums, k):
    res = [dp_quadratic(nums, k), bit_solution(nums, k), seg_solution(nums, k), Solution().maxAlternatingSum(nums, k)]
    if len(nums) <= 12:
        res.append(brute(nums, k))
    return res

for nums, k, exp in examples:
    assert all(v == exp for v in run_all(nums, k)), run_all(nums, k)
print("Ví dụ đề: OK")

Ví dụ đề: OK


In [16]:
# 12.2 ca biên (tự dựng, kiểm chứng bằng vét cạn)
edge = [
    ([7, 7, 7, 7], 1, 7),          # mọi giá trị bằng nhau: chỉ chọn được 1 phần tử
    ([1, 2, 3, 4, 5], 1, 9),        # tăng dần: tốt nhất 4 < 5 → 9
    ([1, 100, 1, 100], 4, 100),     # k = n: chỉ chọn 1 phần tử
    ([1, 100, 1, 100], 2, 101),     # chỉ số cách 2: 1 & 1 hoặc 100 & 100 (bằng nhau) → chọn (0,3) hoặc (1,3)?
    ([2, 1, 2, 1, 2, 1], 1, 9),     # zigzag sẵn: lấy hết
]
for nums, k, exp in edge:
    got = run_all(nums, k)
    assert all(v == got[0] for v in got), got
    print(nums, k, "->", got[0], "| vét cạn:", brute(nums, k))

[7, 7, 7, 7] 1 -> 7 | vét cạn: 7
[1, 2, 3, 4, 5] 1 -> 9 | vét cạn: 9
[1, 100, 1, 100] 4 -> 100 | vét cạn: 100
[1, 100, 1, 100] 2 -> 101 | vét cạn: 101
[2, 1, 2, 1, 2, 1] 1 -> 9 | vét cạn: 9


Ghi chú ca `[1, 100, 1, 100], k = 2`: các cặp chỉ số hợp lệ là `(0,2)`, `(1,3)`, `(0,3)`; `(0,2)` và `(1,3)` có giá trị bằng nhau nên loại, `(0,3)` cho `1 < 100` = `101`. Vét cạn xác nhận.

In [17]:
# 12.3 phản ví dụ cho ý tưởng sai: tham lam "luôn chọn phần tử lớn nhất có thể"
def greedy(nums, k):
    best_single = max(nums)
    return best_single
nums, k = [3, 5, 4, 2, 4], 1
print("tham lam lấy max:", greedy(nums, k), "| đúng:", Solution().maxAlternatingSum(nums, k))
# và ý tưởng sai "chỉ cần k = 1 rồi bỏ qua k"
nums, k = [5, 4, 2], 2
print("bỏ qua k:", Solution().maxAlternatingSum(nums, 1), "| đúng (k=2):", Solution().maxAlternatingSum(nums, k))

tham lam lấy max: 5 | đúng: 14
bỏ qua k: 9 | đúng (k=2): 7


In [18]:
# 12.4 đối chiếu ngẫu nhiên (seed cố định)
rng = random.Random(3915)
for _ in range(500):
    n = rng.randint(1, 11)
    nums = [rng.randint(1, 6) for _ in range(n)]
    k = rng.randint(1, n)
    r = run_all(nums, k)
    assert len(set(r)) == 1, (nums, k, r)
print("500 test nhỏ: OK")
for _ in range(80):
    n = rng.randint(100, 400)
    nums = [rng.randint(1, 50) for _ in range(n)]
    k = rng.randint(1, 20)
    r = run_all(nums, k)
    assert len(set(r)) == 1
print("80 test vừa: OK")

500 test nhỏ: OK


80 test vừa: OK


In [19]:
# 12.5 test lớn đúng ràng buộc
rng = random.Random(7)
big = [rng.randint(1, 10**5) for _ in range(10**5)]
t0 = time.time(); a = Solution().maxAlternatingSum(big, 3); t1 = time.time()
b = seg_solution(big, 3)
print("đáp án:", a, "| khớp segment tree:", a == b, "| < 3s:", t1 - t0 < 3)

đáp án: 1813777901 | khớp segment tree: True | < 3s: True


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Chèn ngay vị trí `i` thay vì `i - k` (quên điều kiện khoảng cách)

In [20]:
def wrong_no_delay(nums, k):
    vals = sorted(set(nums)); m = len(vals)
    rank = {v: i + 1 for i, v in enumerate(vals)}
    bd, bu = BITMax(m), BITMax(m)
    ans = 0
    for x in nums:
        r = rank[x]
        d, u = x + bu.query(m - r), x + bd.query(r - 1)
        bd.update(r, d); bu.update(m + 1 - r, u)      # SAI: chèn ngay, coi như k = 1
        ans = max(ans, d, u)
    return ans

print("sai:", wrong_no_delay([5, 4, 2], 2), "| đúng:", Solution().maxAlternatingSum([5, 4, 2], 2))

sai: 9 | đúng: 7


**Vì sao sai:** chèn ngay cho phép nối `j = i - 1` dù `k = 2`, nên chọn được `5 > 4 < …`? Ở đây ra `9` (`5 + 4`) trong khi chỉ số 0 và 1 cách nhau 1 < k. **Sửa:** chỉ chèn `i - k` khi xử lý `i`.

### Lỗi 2 — Dùng `≤`/`≥` thay vì `<`/`>` (cho phép giá trị bằng nhau)

In [21]:
def wrong_nonstrict(nums, k):
    vals = sorted(set(nums)); m = len(vals)
    rank = {v: i + 1 for i, v in enumerate(vals)}
    bd, bu = BITMax(m), BITMax(m)
    n = len(nums); down, up = [0] * n, [0] * n; ans = 0
    for i, x in enumerate(nums):
        if i >= k:
            j = i - k; rj = rank[nums[j]]
            bd.update(rj, down[j]); bu.update(m + 1 - rj, up[j])
        r = rank[x]
        down[i] = x + bu.query(m + 1 - r)      # SAI: gồm cả hạng bằng r
        up[i] = x + bd.query(r)                # SAI
        ans = max(ans, down[i], up[i])
    return ans

print("sai:", wrong_nonstrict([7, 7, 7], 1), "| đúng:", Solution().maxAlternatingSum([7, 7, 7], 1))

sai: 21 | đúng: 7


### Lỗi 3 — Lẫn lộn vai trò: đáy nối từ đáy, đỉnh nối từ đỉnh

In [22]:
def wrong_roles(nums, k):
    n = len(nums); down, up = [0] * n, [0] * n
    for i in range(n):
        bd = bu = 0
        for j in range(i - k + 1):
            if nums[j] > nums[i]: bd = max(bd, down[j])   # SAI: phải là up[j]
            if nums[j] < nums[i]: bu = max(bu, up[j])     # SAI: phải là down[j]
        down[i], up[i] = nums[i] + bd, nums[i] + bu
    return max(down + up)

print("sai:", wrong_roles([1, 2, 3, 4, 5], 1), "| đúng:", Solution().maxAlternatingSum([1, 2, 3, 4, 5], 1))

sai: 15 | đúng: 9


**Vì sao sai:** nối "đỉnh" sau "đỉnh" nhỏ hơn cho phép dãy tăng `1 < 2 < 3 < 4 < 5` (tổng 15), không còn luân phiên. **Sửa:** đáy hỏi **đỉnh** lớn hơn, đỉnh hỏi **đáy** nhỏ hơn.

### Lỗi 4 — Quên lật trục khi dùng Fenwick cho truy vấn "lớn hơn"

Fenwick chỉ trả lời max **tiền tố**. Nếu truy vấn `query(m) - …` (như với tổng) sẽ sai vì `max` không có phép trừ ngược. Phải lưu ở chỉ số `m + 1 - r` hoặc dùng segment tree.

In [23]:
def wrong_no_flip(nums, k):
    vals = sorted(set(nums)); m = len(vals)
    rank = {v: i + 1 for i, v in enumerate(vals)}
    bd, bu = BITMax(m), BITMax(m)
    n = len(nums); down, up = [0] * n, [0] * n; ans = 0
    for i, x in enumerate(nums):
        if i >= k:
            j = i - k; rj = rank[nums[j]]
            bd.update(rj, down[j]); bu.update(rj, up[j])     # SAI: không lật trục
        r = rank[x]
        down[i] = x + bu.query(m)            # SAI: max toàn bộ, gồm cả giá trị nhỏ hơn
        up[i] = x + bd.query(r - 1)
        ans = max(ans, down[i], up[i])
    return ans

print("sai:", wrong_no_flip([1, 2, 3, 4, 5], 1), "| đúng:", Solution().maxAlternatingSum([1, 2, 3, 4, 5], 1))

sai: 15 | đúng: 9


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **DP dãy con + điều kiện so sánh với phần tử trước** ⇒ "max theo khoảng giá trị" ⇒ Fenwick/segment tree trên trục giá trị (đã nén).
- **Điều kiện khoảng cách chỉ số `≥ k`** ⇒ chèn trễ `i - k`. (Nếu là `≤ k` thì cần **xóa** phần tử cũ ⇒ segment tree hoặc deque đơn điệu.)
- **Luân phiên** ⇒ hai trạng thái (đáy/đỉnh), nối chéo.
- Truy vấn **hậu tố** với Fenwick max ⇒ lật trục `m + 1 - r`.

## 15. Biến thể của bài

1. **Không có điều kiện khoảng cách (k = 1)** và đếm độ dài thay vì tổng: *Wiggle Subsequence* (LC 376) — có lời giải tham lam `O(n)`.
2. **Tổng có dấu xen kẽ** (`+a - b + c …`): *Maximum Alternating Subsequence Sum* (LC 1911) — DP hai trạng thái `O(n)`.
3. **Khoảng cách tối đa `k`** (`i_{t+1} - i_t ≤ k`): cần cửa sổ trượt — segment tree có xóa, hoặc cấu trúc theo khối.
4. **Thêm điều kiện chênh lệch giá trị ≤ d**: truy vấn đoạn giá trị `(x, x + d]` — segment tree (Cách 4) xử lý tự nhiên.

In [24]:
# Biến thể 4: thêm điều kiện |nums[j] - nums[i]| <= d, dùng segment tree trên trục giá trị
def variant_diff(nums, k, d):
    vals = sorted(set(nums)); m = len(vals)
    rk = [bisect_left(vals, x) for x in nums]
    sd, su = SegMax(m), SegMax(m)
    n = len(nums); down, up = [0] * n, [0] * n; ans = 0
    for i in range(n):
        if i >= k:
            j = i - k; sd.update(rk[j], down[j]); su.update(rk[j], up[j])
        x = nums[i]
        hi = bisect_left(vals, x + d + 1)      # giá trị trong (x, x+d]
        lo = bisect_left(vals, x - d)          # giá trị trong [x-d, x)
        down[i] = x + su.query(rk[i] + 1, hi)
        up[i] = x + sd.query(lo, rk[i])
        ans = max(ans, down[i], up[i])
    return ans

def brute_diff(nums, k, d):
    n = len(nums); best = 0
    for mask in range(1, 1 << n):
        idx = [i for i in range(n) if mask >> i & 1]
        if any(idx[t + 1] - idx[t] < k or abs(nums[idx[t + 1]] - nums[idx[t]]) > d for t in range(len(idx) - 1)):
            continue
        v = [nums[i] for i in idx]
        if is_alternating(v):
            best = max(best, sum(v))
    return best

rng = random.Random(15)
for _ in range(300):
    n = rng.randint(1, 10); nums = [rng.randint(1, 8) for _ in range(n)]
    k = rng.randint(1, n); d = rng.randint(0, 5)
    assert variant_diff(nums, k, d) == brute_diff(nums, k, d)
print("Biến thể 4 khớp vét cạn.")

Biến thể 4 khớp vét cạn.


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn | Thử mọi dãy con | `O(2^n n)` | `O(n)` | Kiểm thử |
| 2. DP `O(n²)` | `down/up` duyệt mọi `j` | `O(n²)` | `O(n)` | `n ≤ 3000`, hiểu bài |
| 3. DP + 2 Fenwick | Max tiền tố/hậu tố + chèn trễ | `O(n log n)` | `O(n)` | **Lời giải chuẩn**, nhanh, ngắn |
| 4. DP + segment tree | Max đoạn bất kỳ + chèn trễ | `O(n log n)` | `O(n)` | Khi cần truy vấn đoạn tổng quát |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output).** `maxAlternatingSum([1, 3, 2, 4], 1)` = ?

**Bài 2 (tính tay).** Với `nums = [4, 1, 5, 2, 6]`, `k = 2`, lập bảng `down/up` và tính đáp án.

**Bài 3 (điền TODO).** Viết hàm `longest_alternating(nums, k)` trả về **độ dài** lớn nhất (thay vì tổng).

**Bài 4 (tự viết lại).** Viết lại lời giải chính chỉ dùng **một** segment tree lưu cặp `(max_down, max_up)`.

**Bài 5 (mini challenge).** Điều kiện khoảng cách đổi thành `k ≤ i_{t+1} - i_t ≤ K`. Viết lời giải `O(n log n)` hoặc `O(n · K)` và kiểm chứng.

In [25]:
# Bài 3 — TODO (không gọi hàm này khi chạy notebook)
def longest_alternating(nums, k):
    # TODO: như lời giải chính nhưng cộng 1 thay vì cộng nums[i]
    pass

In [26]:
# Bài 5 — TODO (không gọi hàm này khi chạy notebook)
def alt_sum_window(nums, k, K):
    # TODO: chỉ xét j trong [i-K, i-k]
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Dãy `1 < 3 > 2 < 4` luân phiên. (2) Lấy hết các phần tử. (3) `10`.

**Bài 2.** (1) `j ≤ i - 2`, nên chỉ nối được các chỉ số cách nhau ≥ 2. (2) Các dãy 3 phần tử `0,2,4` là `4 < 5 < 6` — không luân phiên; xét các cặp: `(2,4)`: `5 < 6` → 11, `(0,4)`: 10, `(0,2)`: 9. (3) Đáp án `11` (`up[4] = 6 + down[2] = 6 + 5`).

**Bài 3.** (1) Thay `x +` bằng `1 +`. (2) Giữ nguyên BIT. (3) Đáp án là max mọi trạng thái.

**Bài 4.** (1) Mỗi nút lưu hai giá trị. (2) Truy vấn trả về cặp. (3) Lấy `[0]` cho đáy, `[1]` cho đỉnh.

**Bài 5.** (1) Duyệt `j` từ `max(0, i-K)` tới `i-k`. (2) `O(n·K)` dễ viết. (3) Đối chiếu vét cạn.

## 19. Đáp án tham khảo cho bài tập

In [27]:
# Bài 1, 2
assert Solution().maxAlternatingSum([1, 3, 2, 4], 1) == 10 == brute([1, 3, 2, 4], 1)
b2 = Solution().maxAlternatingSum([4, 1, 5, 2, 6], 2)
assert b2 == brute([4, 1, 5, 2, 6], 2)
print("Bài 1: 10 | Bài 2:", b2)

Bài 1: 10 | Bài 2: 11


In [28]:
# Bài 3
def longest_alternating_sol(nums, k):
    vals = sorted(set(nums)); m = len(vals)
    rank = {v: i + 1 for i, v in enumerate(vals)}
    bd, bu = BITMax(m), BITMax(m)
    n = len(nums); down, up = [0] * n, [0] * n
    for i, x in enumerate(nums):
        if i >= k:
            j = i - k; rj = rank[nums[j]]
            bd.update(rj, down[j]); bu.update(m + 1 - rj, up[j])
        r = rank[x]
        down[i], up[i] = 1 + bu.query(m - r), 1 + bd.query(r - 1)
    return max(down + up)

def brute_len(nums, k):
    n = len(nums); best = 0
    for mask in range(1, 1 << n):
        idx = [i for i in range(n) if mask >> i & 1]
        if all(idx[t + 1] - idx[t] >= k for t in range(len(idx) - 1)) and is_alternating([nums[i] for i in idx]):
            best = max(best, len(idx))
    return best

rng = random.Random(3)
for _ in range(300):
    n = rng.randint(1, 10); nums = [rng.randint(1, 6) for _ in range(n)]; k = rng.randint(1, n)
    assert longest_alternating_sol(nums, k) == brute_len(nums, k)
print("Bài 3 OK")

Bài 3 OK


In [29]:
# Bài 4
class SegPair:
    def __init__(self, n):
        self.size = 1
        while self.size < n: self.size *= 2
        self.t = [(0, 0)] * (2 * self.size)
    def update(self, pos, d, u):
        pos += self.size
        a, b = self.t[pos]; self.t[pos] = (max(a, d), max(b, u)); pos //= 2
        while pos:
            l, r = self.t[2 * pos], self.t[2 * pos + 1]
            self.t[pos] = (max(l[0], r[0]), max(l[1], r[1])); pos //= 2
    def query(self, lo, hi):
        a = b = 0; lo += self.size; hi += self.size
        while lo < hi:
            if lo & 1: a = max(a, self.t[lo][0]); b = max(b, self.t[lo][1]); lo += 1
            if hi & 1: hi -= 1; a = max(a, self.t[hi][0]); b = max(b, self.t[hi][1])
            lo //= 2; hi //= 2
        return a, b

def sol_one_tree(nums, k):
    vals = sorted(set(nums)); m = len(vals)
    rk = [bisect_left(vals, x) for x in nums]
    st = SegPair(m); n = len(nums); down, up = [0] * n, [0] * n; ans = 0
    for i in range(n):
        if i >= k:
            j = i - k; st.update(rk[j], down[j], up[j])
        down[i] = nums[i] + st.query(rk[i] + 1, m)[1]
        up[i] = nums[i] + st.query(0, rk[i])[0]
        ans = max(ans, down[i], up[i])
    return ans

rng = random.Random(4)
for _ in range(300):
    n = rng.randint(1, 30); nums = [rng.randint(1, 10) for _ in range(n)]; k = rng.randint(1, n)
    assert sol_one_tree(nums, k) == Solution().maxAlternatingSum(nums, k)
print("Bài 4 OK")

Bài 4 OK


In [30]:
# Bài 5
def alt_sum_window_sol(nums, k, K):
    n = len(nums); down, up = [0] * n, [0] * n
    for i in range(n):
        bd = bu = 0
        for j in range(max(0, i - K), i - k + 1):
            if nums[j] > nums[i]: bd = max(bd, up[j])
            elif nums[j] < nums[i]: bu = max(bu, down[j])
        down[i], up[i] = nums[i] + bd, nums[i] + bu
    return max(down + up)

def brute_window(nums, k, K):
    n = len(nums); best = 0
    for mask in range(1, 1 << n):
        idx = [i for i in range(n) if mask >> i & 1]
        if all(k <= idx[t + 1] - idx[t] <= K for t in range(len(idx) - 1)) and is_alternating([nums[i] for i in idx]):
            best = max(best, sum(nums[i] for i in idx))
    return best

rng = random.Random(5)
for _ in range(300):
    n = rng.randint(1, 10); nums = [rng.randint(1, 7) for _ in range(n)]
    k = rng.randint(1, n); K = rng.randint(k, n)
    assert alt_sum_window_sol(nums, k, K) == brute_window(nums, k, K)
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
rank = nén(nums); m = số giá trị
for i, x in enumerate(nums):
    if i >= k:                         # chèn trễ
        j = i - k
        bit_down.update(rank[nums[j]], down[j])
        bit_up.update(m + 1 - rank[nums[j]], up[j])   # lật trục
    r = rank[x]
    down[i] = x + bit_up.query(m - r)  # đáy hỏi ĐỈNH lớn hơn
    up[i]   = x + bit_down.query(r - 1)  # đỉnh hỏi ĐÁY nhỏ hơn
ans = max(down + up)
```

**Checklist:**

- [ ] Chèn **trễ** đúng vị trí `i - k` (không chèn `i`)?
- [ ] So sánh **ngặt** (`r - 1` và `m - r`)?
- [ ] Đáy nối từ đỉnh, đỉnh nối từ đáy?
- [ ] Lật trục cho truy vấn "lớn hơn" với Fenwick?
- [ ] Dãy 1 phần tử được tính (giá trị khởi tạo `0` + `x`)?